# 키워드 채널 적합도 EDA — 인스타(바이럴) vs POS(매출)

> 방법론: `docs/causal_intervention_and_metapath.md` §2 (관찰 게이트 → 인과 Δ 검증)
> 모델: **v2_sweepA** (현 서빙) · 대시보드와 동일 `score_concept` 프리미티브 · 노이즈 플로어 ±0.01 통일

## 무엇을 하나
1. **배타 성공군 분할**: POS 단독성공(`성공_소스=='POS'`) vs 인스타 단독성공(`인스타/CU/GS25`). **POS+인스타(둘 다 성공)는 채널 판별 노이즈라 제외.**
2. **관찰 게이트(커버리지)**: 키워드가 POS 성공군에 ≥3 / 인스타 성공군에 ≥1 등장하나. (killer/mine과 동일한 빈도≥3 base 위에서)
3. **인과 Δ (ablation)**: 그 키워드를 *보유한* 성공작에서 키워드를 **빼봤을 때** 성공확률 하락폭. 채널별로 평균.
   - `Δ_pos(k)`, `Δ_insta(k)` → **margin = Δ_insta − Δ_pos** (±0.01 노이즈 플로어로 끊음)
4. **태그**: POS형 / 인스타형 / 범용(양쪽) / 채널미정. 인스타 1~2개 표본은 **저신뢰 플래그**.

## 이 노트북의 진짜 목적
뽑힌 키워드가 **납득되는지**(§5) + 이 채널 태그가 기존 killer/mine/매개 태그와 **직교한 새 정보인지**(§6)를 보고, **대시보드에 녹일지 결정**한다.

In [1]:
# --- Setup ---
import os, sys
from collections import defaultdict
# repo 루트 자동 탐색 (cwd가 notebooks/든 repo 루트든 무관) — src/eval/md/engine.py 마커로 상향 검색
ROOT = os.path.abspath(os.getcwd())
for _ in range(8):
    if os.path.exists(os.path.join(ROOT, "src", "eval", "md", "engine.py")):
        break
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        raise RuntimeError("repo 루트를 못 찾음 — src/eval/md/engine.py 기준")
    ROOT = parent
if ROOT not in sys.path: sys.path.insert(0, ROOT)
os.chdir(ROOT)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False

from src.eval.md.engine import MDEngine, EngineConfig, PK_MAIN
print("repo:", ROOT)

c:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


repo: c:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework


## ⚙️ 0. 파라미터 (튜닝 다이얼)

- `POS_FLOOR=3` / `INSTA_FLOOR=1`: 채널 게이트 (인스타는 표본이 얇어 1로 푼다 — 소실 vs 정밀 트레이드오프).
- `TOTAL_FLOOR=3`: killer/mine과 동일 base (소표본 거품 제거).
- `NOISE=0.01`: 단일 노이즈 플로어 — 대시보드 pair synergy / 전역 분류와 동일 기준선.
- `LOWCONF_INSTA=3`: 인스타 보유 < 이 값이면 저신뢰 플래그(버리지 않음).

In [2]:
POS_FLOOR    = 3
INSTA_FLOOR  = 1
TOTAL_FLOOR  = 3
NOISE        = 0.01
LOWCONF_INSTA= 3
MODEL        = "v2_sweepA"   # 현 서빙. 비교하려면 "exp47".

## 1. 단일 추론 + 채널 배타 마스크

`succ_src` 로 성공 제품을 배타 분할. POS+인스타(둘 다)는 양쪽에서 뺀다.

In [3]:
cfg = EngineConfig.v2_sweepA() if MODEL == "v2_sweepA" else EngineConfig.exp47()
eng = MDEngine(cfg).run_single_inference(); eng.build_mass()

y   = eng.cache["y"]
src = eng.cache["succ_src"]
K   = eng.cache["K"]

INSTA_SRC = ["인스타", "CU_인스타", "GS25_인스타"]
pos_idx   = np.where((y == 1) & (src == "POS"))[0]                  # POS 단독성공
insta_idx = np.where((y == 1) & np.isin(src, INSTA_SRC))[0]        # 인스타 단독성공
both_n    = int(((y == 1) & (src == "POS+인스타")).sum())          # 제외 대상
print(f"POS 단독성공 {len(pos_idx)} / 인스타 단독성공 {len(insta_idx)} / POS+인스타(제외) {both_n}")

POS 단독성공 614 / 인스타 단독성공 265 / POS+인스타(제외) 318


## 2. 관찰 게이트 — 채널별 보유수 + 커버리지 (소실률 리포트)

has_kw 엣지로 키워드별 (전체 / POS성공 / 인스타성공) 보유 제품 수를 센다.

In [4]:
ei = eng.cache["eidx"][PK_MAIN].numpy()   # [2, E]  (product, keyword)

def holder_count(prod_idx):
    m = np.isin(ei[0], prod_idx)
    return np.bincount(ei[1][m], minlength=K)

total_cnt = np.bincount(ei[1], minlength=K)
pos_cnt   = holder_count(pos_idx)
insta_cnt = holder_count(insta_idx)

names = np.array([eng.kw_name(k) for k in range(K)])
df = pd.DataFrame({"keyword": names, "total": total_cnt, "pos": pos_cnt, "insta": insta_cnt})
df = df[df["total"] >= TOTAL_FLOOR].reset_index(drop=True)   # killer/mine base

pos_ok = df["pos"] >= POS_FLOOR
ins_ok = df["insta"] >= INSTA_FLOOR
n = len(df)
print(f"base(총빈도>={TOTAL_FLOOR}) 키워드: {n}")
print(f"  POS 키워드(POS>={POS_FLOOR}):    {int(pos_ok.sum())} ({pos_ok.mean()*100:.1f}%)")
print(f"  인스타 키워드(insta>={INSTA_FLOOR}): {int(ins_ok.sum())} ({ins_ok.mean()*100:.1f}%)")
print(f"   ├ both:    {int((pos_ok&ins_ok).sum())}")
print(f"   ├ POS만:   {int((pos_ok&~ins_ok).sum())}")
print(f"   └ insta만: {int((~pos_ok&ins_ok).sum())}")
print(f"  채널 판정(둘 중 하나라도): {int((pos_ok|ins_ok).sum())} ({(pos_ok|ins_ok).mean()*100:.1f}%)")
print(f"  소실(둘 다 미달):          {int((~pos_ok&~ins_ok).sum())} ({(~pos_ok&~ins_ok).mean()*100:.1f}%)")

base(총빈도>=3) 키워드: 1100
  POS 키워드(POS>=3):    290 (26.4%)
  인스타 키워드(insta>=1): 391 (35.5%)
   ├ both:    178
   ├ POS만:   112
   └ insta만: 213
  채널 판정(둘 중 하나라도): 503 (45.7%)
  소실(둘 다 미달):          597 (54.3%)


## 3. 인과 Δ (ablation) — 채널별 기여도

**핵심 단계.** 키워드를 *보유한* 성공작에서 그 키워드를 빼보고(`score_concept`로 가상 재구성),
원래 대비 성공확률 하락폭 `contrib = prob(보유) − prob(제거)` 를 채널별로 평균한다.

- 정확도: 같은 제품의 `full`과 `ablate_k`를 **한 청크에 넣어** 동시 forward → 차분 기준 1차 상쇄(엔진 docstring).
- 대시보드 `combo`/`classify`와 동일 프리미티브 → 수치 정합.
- 후보(`cand`)는 게이트 통과 키워드만 — 비용 절약.

In [5]:
# 게이트 통과(둘 중 하나라도) 키워드만 Δ 계산 대상
cand_set = set(np.where((total_cnt >= TOTAL_FLOOR))[0].tolist())
cand_set = {k for k in cand_set if pos_cnt[k] >= POS_FLOOR or insta_cnt[k] >= INSTA_FLOOR}

HASIP = ("product", "has_ip", "ip")
have_ip = HASIP in eng.cache["eidx"]
union_prod = sorted(set(pos_idx.tolist()) | set(insta_idx.tolist()))
pk_set = {p: eng.product_keywords(p) for p in union_prod}
ip_set = {p: (eng.product_keywords(p, HASIP) if have_ip else []) for p in union_prod}

def product_contribs(p):
    '''제품 p에서 보유 후보 키워드를 각각 빼본 contrib dict {k: prob(full)-prob(full\\k)}.'''
    kws, ips = pk_set[p], ip_set[p]
    targets = [k for k in kws if k in cand_set]
    if not targets:
        return {}
    concepts = [(kws, ips)] + [([x for x in kws if x != k], ips) for k in targets]
    s = eng.score_concept_batch(concepts, chunk_size=len(concepts))  # 단일 청크 → 상쇄 최대
    base = s[0]
    return {k: float(base - s[i + 1]) for i, k in enumerate(targets)}

def channel_delta(prod_idx):
    acc = defaultdict(list)
    for p in prod_idx:
        for k, c in product_contribs(int(p)).items():
            acc[k].append(c)
    d = {k: float(np.mean(v)) for k, v in acc.items()}
    nn = {k: len(v) for k, v in acc.items()}
    return d, nn

import time
t0 = time.time()
d_pos, _   = channel_delta(pos_idx)
d_insta, _ = channel_delta(insta_idx)
print(f"Δ 계산 완료: POS {len(d_pos)}키 / insta {len(d_insta)}키  ({time.time()-t0:.0f}s)")

Δ 계산 완료: POS 395키 / insta 391키  (51s)


## 4. 태그 분류 — margin 기준

- both(POS≥3 & insta≥1): `margin = Δ_insta − Δ_pos` → `>+0.01` 인스타형 / `<−0.01` POS형 / 그 사이 **범용**.
- 한쪽만 통과: 그 채널로 직판정 (반대쪽은 증거 0 = 측정 불가).
- 인스타 보유 < `LOWCONF_INSTA` 면 **저신뢰** 플래그 (드롭하지 않음).

In [6]:
rows = []
for _, r in df.iterrows():
    k = int(np.where(names == r["keyword"])[0][0])
    dp = d_pos.get(k, np.nan); di = d_insta.get(k, np.nan)
    p_ok = r["pos"] >= POS_FLOOR; i_ok = r["insta"] >= INSTA_FLOOR
    margin = (di - dp) if (p_ok and i_ok) else np.nan
    if p_ok and i_ok:
        tag = "인스타형" if margin > NOISE else ("POS형" if margin < -NOISE else "범용")
    elif p_ok:
        tag = "POS형"
    elif i_ok:
        tag = "인스타형"
    else:
        tag = "채널미정"
    lowconf = (tag == "인스타형") and (r["insta"] < LOWCONF_INSTA)
    rows.append(dict(keyword=r["keyword"], tag=tag, total=int(r["total"]),
                     pos_n=int(r["pos"]), insta_n=int(r["insta"]),
                     d_pos=round(dp, 4) if not np.isnan(dp) else None,
                     d_insta=round(di, 4) if not np.isnan(di) else None,
                     margin=round(margin, 4) if not np.isnan(margin) else None,
                     저신뢰=lowconf))
res = pd.DataFrame(rows)
print(res["tag"].value_counts())
print(f"\n저신뢰(인스타 보유<{LOWCONF_INSTA}) 인스타형: {int(res['저신뢰'].sum())}")
res.head(20)

tag
채널미정    597
인스타형    306
POS형    154
범용       43
Name: count, dtype: int64

저신뢰(인스타 보유<3) 인스타형: 188


,keyword,tag,total,pos_n,insta_n,d_pos,d_insta,margin,저신뢰
0,1인,채널미정,18,2,0,NaN,NaN,NaN,False
1,?,범용,26,3,2,-0.0366,-0.0351,0.0015,False
2,ABC,채널미정,3,0,0,NaN,NaN,NaN,False
3,BBQ,채널미정,5,1,0,NaN,NaN,NaN,False
4,CAFE25,인스타형,8,0,3,NaN,-0.0073,NaN,False
5,CJ,인스타형,25,0,2,NaN,-0.0330,NaN,True
6,HEYROO,인스타형,12,0,2,NaN,-0.0448,NaN,True
7,ICE,채널미정,4,0,0,NaN,NaN,NaN,False
8,JD,채널미정,4,0,0,NaN,NaN,NaN,False
9,LA,채널미정,3,0,0,NaN,NaN,NaN,False


## 5. 방법 검증 ① — 전역 채널 수치가 납득되나?

라이브로 노드에 붙일 채널 수치(ablation Δ)가 직관과 맞는지 먼저 본다. (가설: POS형=식감·내실 / 인스타형=트렌드·디저트)
> 채널 라벨은 정적 리스트가 아니라 **gtag(killer/mine)처럼 전역 1회 산출(§3 배치) → 서브네트 노드에 부착**할 속성이다. 여기선 그 산출값이 타당한지 본다.

In [7]:
def show(tag, by, asc, nshow=15):
    sub = res[res["tag"] == tag].sort_values(by, ascending=asc)
    print(f"=== {tag} ({len(res[res['tag']==tag])}개) ===")
    return sub.head(nshow)[["keyword","pos_n","insta_n","d_pos","d_insta","margin","저신뢰"]]
display(show("인스타형", "insta_n", False))
display(show("POS형",   "pos_n",   False))
display(show("범용",     "total",   False))

=== 인스타형 (306개) ===


,keyword,pos_n,insta_n,d_pos,d_insta,margin,저신뢰
215,디저트,59,78,-0.0548,-0.0338,0.0210,False
454,빵,62,53,-0.0098,0.0085,0.0183,False
24,간편,13,47,-0.0639,-0.0456,0.0183,False
626,야식,112,39,-0.0174,-0.0023,0.0150,False
822,초코,79,34,-0.0598,-0.0281,0.0317,False
695,우유,37,32,-0.0228,-0.0083,0.0146,False
880,콜라,1,27,-0.1032,-0.0521,NaN,False
64,과자,53,27,-0.0163,0.0081,0.0243,False
575,시즌,1,26,0.0149,0.0322,NaN,False
46,고기,41,22,-0.0362,0.0012,0.0374,False


=== POS형 (154개) ===


,keyword,pos_n,insta_n,d_pos,d_insta,margin,저신뢰
52,고소,122,22,0.0345,0.0236,-0.0108,False
795,짭조름함,73,19,0.1047,0.0627,-0.0421,False
577,식감,66,0,0.0706,NaN,NaN,False
538,술,41,10,-0.0082,-0.0374,-0.0292,False
167,담백,36,0,-0.0222,NaN,NaN,False
789,진함,30,1,-0.0280,-0.1039,-0.0759,False
920,탱글함,22,1,0.0981,0.0615,-0.0366,False
233,라면,22,15,0.0640,0.0445,-0.0195,False
63,과일,22,11,-0.0185,-0.0511,-0.0326,False
154,단백질,20,0,-0.0034,NaN,NaN,False


=== 범용 (43개) ===


,keyword,pos_n,insta_n,d_pos,d_insta,margin,저신뢰
22,간식,451,121,0.0009,0.0043,0.0035,False
159,달콤,236,66,0.0371,0.0392,0.0021,False
415,부드러움,166,19,0.0232,0.0329,0.0097,False
581,식사,78,25,-0.0390,-0.0371,0.0019,False
903,크림,62,55,0.0036,0.0120,0.0084,False
300,매콤,71,24,-0.0001,0.0089,0.0090,False
376,밥,37,17,-0.0363,-0.0280,0.0083,False
482,상큼,73,11,0.0266,0.0345,0.0078,False
796,쫀득함,43,20,0.0118,0.0188,0.0070,False
823,촉촉,75,9,0.0869,0.0910,0.0041,False


## 6. ★ 방법 검증 ② — 라이브 서브네트워크 내 채널 구분 (핵심)

대시보드 사용 흐름 그대로: **시드 주입 → `subnet.build_subnetwork`로 양의-시너지 서브네트 생성 → 그 안의 키워드에 채널 라벨 부착.**
MD는 이 서브네트에서 5개를 뽑아 **인스타형=마케팅 카피 / POS형=제품 활용**으로 배분한다.

여기서 검증할 것:
- 서브네트 안의 키워드들이 **채널로 깔끔히 갈리나** (예시 납득도).
- **서브네트 내 채널미정 비율**이 낮은가 (전역 45.7%보다 높아야 라이브 부착 가치 ↑ — 양의 시너지 키워드는 빈도가 높아 더 잘 잡힐 것이라는 가설 검증).
- 비용 — gtag처럼 로드 1회 산출이라 부착은 dict 조회(≈0).

In [8]:
from src.eval.md.combo import _ConceptCache
from src.eval.md import subnet as SN
sc = _ConceptCache(eng)
chan = dict(zip(res["keyword"], res["tag"]))
lowc = dict(zip(res["keyword"], res["저신뢰"]))

SEEDS = ["마라", "로제", "흑임자", "단백질", "고창", "약과"]
cov_rows = []
for seed in SEEDS:
    if eng.seed_to_idx(seed) is None:
        print(f"[{seed}] 그래프에 없음"); continue
    net = SN.build_subnetwork(eng, seed, sc=sc)
    if net.get("error"):
        print(f"[{seed}] {net['error']}"); continue
    kws = [n["label"] for n in net["nodes"] if n["type"] in ("rail","trend","basket","ip2")]
    kws = [k for k in dict.fromkeys(kws) if k != seed]
    ins = [f"{k}{'⚠' if lowc.get(k) else ''}" for k in kws if chan.get(k)=="인스타형"]
    pos = [k for k in kws if chan.get(k)=="POS형"]
    uni = [k for k in kws if chan.get(k)=="범용"]
    und = [k for k in kws if chan.get(k,"채널미정")=="채널미정"]
    ndet = len(kws)-len(und)
    print(f"\n● [{seed}] 서브네트 키워드 {len(kws)} | 채널판정 {ndet} ({ndet/max(len(kws),1)*100:.0f}%)")
    print(f"   📷 인스타형(카피): {ins}")
    print(f"   🔴 POS형(제품):    {pos}")
    print(f"   ⚪ 범용:           {uni}")
    print(f"   ❔ 채널미정:        {und}")
    cov_rows.append(dict(seed=seed, n=len(kws), 판정=ndet, 인스타=len(ins), POS=len(pos), 범용=len(uni), 미정=len(und)))

cov = pd.DataFrame(cov_rows)
if len(cov):
    display(cov)
    print(f"\n서브네트 평균 채널판정률: {cov['판정'].sum()/cov['n'].sum()*100:.0f}%  (전역 base 45.7% 대비)")


● [마라] 서브네트 키워드 46 | 채널판정 25 (54%)
   📷 인스타형(카피): ['탕⚠', '롯데', '국물', '과자', '로제⚠', '분식⚠', '소금', '핫도그⚠', '푸딩', '오뚜기⚠', '맛', '간편', '국수', '면', '파스타', '캠핑⚠', '빵']
   🔴 POS형(제품):    ['누들', '짭조름함', '라면', '육수', '견과류', '신라면']
   ⚪ 범용:           ['콜라보', '쫄깃함']
   ❔ 채널미정:        ['동파육', '딤섬', '샹궈', '피', '순대', '큰컵', '폭탄', '폼폼푸린', '리본', '판타지', '게임', '혼합', '피카츄', '순함', '얼라이브', '열', '냉우동', '자가제면', '경주', '하리보', '참깨']

● [로제] 서브네트 키워드 40 | 채널판정 25 (62%)
   📷 인스타형(카피): ['분식⚠', '소금', '핫도그⚠', '국수', '면', '파스타', '캠핑⚠', '과자', '팥', '기념일⚠', '볶음', '포테이토⚠', '앙금⚠', '빵']
   🔴 POS형(제품):    ['짭조름함', '누들', '탄산', '견과류', '빈티지', '구수함', '설탕']
   ⚪ 범용:           ['쫄깃함', '달콤', '부드러움', '간식']
   ❔ 채널미정:        ['순대', '큰컵', '폭탄', '냉우동', '자가제면', '경주', '하리보', '참깨', '아기자기', '코코아', '유니', '홍콩', '트로트', '다채로움', '오레오']

● [흑임자] 서브네트 키워드 40 | 채널판정 27 (68%)
   📷 인스타형(카피): ['샐러드⚠', '케이크', '바삭', '시즌', '사과⚠', '면', '포테이토⚠', '위스키⚠', '바닐라⚠', '도시락', '맛', '커스터드', '핫⚠', '막걸리⚠', '호떡⚠', '카레⚠']
   🔴 POS형(제품):    ['묵직함', '저당', '견과류', '한정', '담백', '

,seed,n,판정,인스타,POS,범용,미정
0,마라,46,25,17,6,2,21
1,로제,40,25,14,7,4,15
2,흑임자,40,27,16,8,3,13
3,단백질,48,29,17,6,6,19
4,고창,20,14,7,3,4,6
5,약과,35,21,9,8,4,14



서브네트 평균 채널판정률: 62%  (전역 base 45.7% 대비)


In [9]:
# 라이브 비용 체크 — combo 서브네트 1회. 채널 부착은 gtag처럼 dict 조회라 추가비용 ≈0.
import time
t0 = time.time(); _ = SN.build_subnetwork(eng, "마라", sc=sc)
print(f"combo 서브네트 1회: {time.time()-t0:.2f}s  (+채널 부착 ≈0 → combo와 동일하게 라이브 가능)")

combo 서브네트 1회: 0.06s  (+채널 부착 ≈0 → combo와 동일하게 라이브 가능)


## 7. 의사결정 — 라이브 `ctag`로 녹일까?

판단 재료:
1. **방법 타당성**(§5) — 전역 채널 수치가 직관과 맞나.
2. **서브네트 커버리지**(§6) — MD가 실제 보는 서브네트에서 채널미정 비율이 낮은가. ★ 이게 결정의 핵심 (전역 45.7%가 아니라 *서브네트 내* 판정률).
3. **직교성**(아래) — 채널 태그가 killer/mine/매개와 다른 새 정보인가.
4. **비용** — gtag처럼 로드 1회 → 라이브 부착 ≈0 (§6에서 확인).

→ §6 서브네트 커버리지가 충분 + §5 납득되면: `classify_channel_live`(로드 1회 캐시) + `combo_serve.build_seed`에서 노드 `ctag` 부착 = **gtag와 완전히 동일한 패턴**으로 연동. (single click = ctag 채널 / two click = 기존 synergy 그대로)

In [10]:
# 직교성 — 기존 전역 분류와 교차 (새 정보인지)
from src.eval.md.classify import classify_keywords_live
gtags = classify_keywords_live(eng)
res["전역태그"] = res["keyword"].map(lambda x: gtags.get(x, "neutral"))
print("채널 태그 × 전역 태그 교차표:")
display(pd.crosstab(res["tag"], res["전역태그"]))

# 검증·연동 판단 자료 저장 (이 CSV가 산출물이 아니라, 라이브 계산법 검증 근거)
OUT = f"experiments/results/md_prescription/{MODEL}/channel_fit"
os.makedirs(OUT, exist_ok=True)
res.to_csv(os.path.join(OUT, "channel_fit_keywords.csv"), index=False, encoding="utf-8-sig")
print("saved:", os.path.join(OUT, "channel_fit_keywords.csv"))

채널 태그 × 전역 태그 교차표:


전역태그,hub,killer,mine,neutral,매개
tag,,,,,
POS형,11,5,8,105,25
범용,7,0,3,27,6
인스타형,35,10,19,204,38
채널미정,17,4,35,533,8


saved: experiments/results/md_prescription/v2_sweepA/channel_fit\channel_fit_keywords.csv
